[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/lab12_privacy_governance.ipynb) · [Course home](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/00_course_home.ipynb)

**Google Colab version.** **Before you start: File → Save a copy in Drive.** Work only in your copy (its title starts with *Copy of*); changes to this original are not saved. Rename your copy `Lab 12 — <your last name>` (click the title at the top left).

**To submit:** **Runtime → Run all** and check that every cell ran, then **File → Save**. Click
**Share** (top right), add **steven.elliott@student.fairfield.edu** as **Editor**, and send. Colab keeps your saved versions
under **File → Revision history**; **File → Save and pin revision** keeps one permanently, which is how you
mark a submission.

**Coming back later?** Colab forgets variables and downloaded files when the runtime disconnects (after about 90 minutes idle). Open your copy and click **Runtime → Run all**.

# Lab 12 — Privacy, ethics, and governance
*Week 12. Baseline: sections 1–5. Powerups: section 6.*

Every number you publish about patients is a promise that nobody can be picked out of it. This week you
keep that promise two ways. First, **small-cell suppression** on a real CMS count table, including the
second round of suppression that stops a reader from recovering a hidden cell by subtraction. Second,
**de-identification** of a patient-level file under the HIPAA Safe Harbor rule, followed by a
**k-anonymity** check that shows what Safe Harbor does not promise.

The patient file in sections 4 and 5 is **synthetic**. The notebook makes it up with a fixed random seed.
No real person is in it.

**How labs work.** Save a copy in Drive, work in your copy, share it back with the instructor
when you submit. Every lab has a **baseline** (everyone finishes it in the session) and
**powerups** (optional depth). AI assistants may write code for you; **you own every line** and
may be asked to explain any of it. Cells marked `# TODO` are yours to fill in.

In [ ]:
# --- Course setup: run this cell first -------------------------------------------
# Loaders for the public datasets this course uses. The canonical copy of this code
# is labs/course_data.py in the course repo; you will read it line by line in week 3.
import io, json, urllib.request, urllib.parse
from pathlib import Path
import pandas as pd
import duckdb

CACHE = Path("data"); CACHE.mkdir(exist_ok=True)

def sql(query: str) -> pd.DataFrame:
    """Run SQL over any DataFrame in this notebook by its variable name (DuckDB)."""
    return duckdb.sql(query).df()

def _get(url, params=None, timeout=120) -> bytes:
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "healthcare-analytics-course/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- CMS Care Compare (data.cms.gov/provider-data) --------------------------------
# File names change at every CMS refresh; dataset identifiers do not. We ask the
# catalog for the current CSV URL every time instead of hard-coding a file name.
CMS_CATALOG = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
CMS_IDS = {
    "hospital_general_info": "xubh-q36u",   # one row per hospital
    "hrrp":                  "9n3s-kdb3",   # Hospital Readmissions Reduction Program, one row per hospital x measure
    "unplanned_visits":      "632h-zaca",   # readmissions / ED visits, one row per hospital x measure
    "complications_deaths":  "ynj2-r877",
    "timely_effective":      "yv7e-xc69",
}

def cms_csv_url(identifier: str) -> str:
    # Ask for the one dataset by id; the full keyword search can take minutes or time out.
    item = json.loads(_get(f"{CMS_CATALOG}/{identifier}", timeout=30))
    for d in item.get("distribution", []):
        if d.get("mediaType") == "text/csv":
            return d["downloadURL"]
    raise KeyError(f"No CSV distribution found for CMS dataset {identifier}")

# Course mirror: a snapshot of every CMS file the labs use, for when CMS is down or rate-limits
# a classroom downloading the same file at once. The live source is always tried first.
COURSE_DATA_BASE = "https://raw.githubusercontent.com/stevenelliottjr/healthcare-analytics-course-data/main/data"

def _fetch_or_mirror(path: Path, fetch, mirror_file: str) -> None:
    """Save `path` from the live source; if that fails, from the course mirror."""
    try:
        path.write_bytes(fetch())
    except Exception as e:
        print(f"Live download failed ({type(e).__name__}); using the course mirror for {path.name}.")
        path.write_bytes(_get(f"{COURSE_DATA_BASE}/{mirror_file}", timeout=300))

def load_cms(name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download (once) and read a Care Compare file. Everything is read as TEXT on purpose:
    IDs keep their leading zeros and 'Not Available' stays visible instead of silently
    becoming NaN. Converting columns to numbers is your job, and part of the lesson."""
    path = CACHE / f"cms_{name}.csv"
    if not path.exists():
        _fetch_or_mirror(path, lambda: _get(cms_csv_url(CMS_IDS[name]), timeout=300), f"cms/cms_{name}.csv")
    kw = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")
    kw.update(read_csv_kwargs)
    return pd.read_csv(path, **kw)

def to_num(s: pd.Series) -> pd.Series:
    """Text -> number; anything that is not a number ('Not Available', 'N/A', '') becomes NaN."""
    return pd.to_numeric(s, errors="coerce")

# ---- CDC PLACES, county release (Socrata API on data.cdc.gov) ---------------------
PLACES_COUNTY = "https://data.cdc.gov/resource/swc5-untb.json"
PLACES_SELECT = ("year,stateabbr,statedesc,locationname,locationid,categoryid,measureid,measure,"
                 "data_value,low_confidence_limit,high_confidence_limit,totalpopulation")

def load_places(measure_ids, data_value_type="Age-adjusted prevalence", state=None, page=50000) -> pd.DataFrame:
    """County-level model-based prevalence estimates (percent of adults). locationid is the
    5-digit county FIPS. Note: Connecticut reports *planning regions*, not counties, since 2022."""
    where = "measureid in(" + ",".join(f"'{m}'" for m in measure_ids) + f") AND data_value_type='{data_value_type}'"
    if state:
        where += f" AND stateabbr='{state}'"
    frames, offset = [], 0
    while True:
        chunk = pd.DataFrame(json.loads(_get(PLACES_COUNTY, {"$where": where, "$select": PLACES_SELECT,
                                                               "$limit": page, "$offset": offset})))
        frames.append(chunk)
        if len(chunk) < page:
            break
        offset += page
    df = pd.concat(frames, ignore_index=True)
    for c in ["data_value", "low_confidence_limit", "high_confidence_limit", "totalpopulation"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ---- Week 4 powerups (links checked 2026-10-26; 2022 is the latest SVI release) ----
SVI_COUNTY_URL = "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv"
def load_svi_county() -> pd.DataFrame:
    """CDC/ATSDR Social Vulnerability Index 2022, one row per county.
    Key columns: FIPS, STATE, COUNTY, E_TOTPOP, RPL_THEMES (overall percentile 0-1),
    RPL_THEME1..RPL_THEME4. The value -999 means 'not available'."""
    path = CACHE / "svi_2022_county.csv"
    if not path.exists():
        path.write_bytes(_get(SVI_COUNTY_URL, timeout=300))
    return pd.read_csv(path, dtype={"FIPS": str}).replace(-999, pd.NA)

CHR_URL = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
def load_county_health_rankings() -> pd.DataFrame:
    """County Health Rankings analytic file. Row 0 holds readable labels and row 1 the variable
    codes; we skip the labels so the codes (fipscode, v001..v1xx) become the column names.
    The data dictionary on the CHR site maps each v-code to a measure."""
    path = CACHE / "chr_analytic_2025_v3.csv"
    if not path.exists():
        path.write_bytes(_get(CHR_URL, timeout=300))
    return pd.read_csv(path, skiprows=[0], dtype={"fipscode": str, "statecode": str, "countycode": str})

print("course setup ready")


## 1. The CMS rule, in CMS's own data
CMS's cell size suppression policy, as ResDAC (CMS's research data assistance center) states it
(current version dated 01/26/2024, https://resdac.org/articles/cms-cell-size-suppression-policy):

> "The policy stipulates that no cell (e.g. admissions, discharges, patients, services, etc.) containing a
> value of 1 to 10 can be reported directly. A value of zero does not violate the minimum cell size
> policy. In addition, no cell can be reported that allows a value of 1 to 10 to be derived from other
> reported cells or information."

The policy is written for researchers using CMS files, but CMS applies the same rule to what it publishes.
You met it in week 2: the HRRP file's `Number of Readmissions` column.

**Baseline:** report how many HRRP rows say `Too Few to Report`, and the **smallest number** CMS actually
shows in that column. One sentence: how does that number match the policy?

In [ ]:
hrrp = load_cms("hrrp")
n_readm = hrrp["Number of Readmissions"]
print(n_readm.value_counts().head(4))
# TODO: how many rows say "Too Few to Report"?
# TODO: of the rows that hold a number (to_num), what is the smallest value shown?


**What the smallest number tells you:** *(edit this cell)*

## 2. A count table, and the first round of suppression
Below is a real table built from *Hospital General Information*: hospitals in New England by state and
type. These are counts of **hospitals**, not patients, so they are not actually sensitive. We use them
because they are public and have the mix of big and small cells that patient tables have. The mechanics
are identical for patient counts.

A published table shows every cell **and** the row and column totals. **Primary suppression** hides every
cell from 1 to 10. Zeros stay.

**Baseline:**
1. Make `primary`, a True/False table the same shape as `table`, True where the cell must be hidden.
   Report how many of the 24 cells it hides.
2. Run the attacker, `recover(table, primary)`. It plays a reader who has the published table: any row or
   column with exactly **one** hidden cell gives that cell away (the total minus the cells you can see).
   Report which cell it recovers, and its value.

In [ ]:
hgi = load_cms("hospital_general_info")
NEW_ENGLAND = ["CT", "MA", "ME", "NH", "RI", "VT"]
TYPE = {"Acute Care Hospitals": "Acute care", "Critical Access Hospitals": "Critical access",
        "Psychiatric": "Psychiatric"}            # everything else (VA, DoD, children's, ...) -> Other
ne = hgi[hgi["State"].isin(NEW_ENGLAND)].copy()
ne["type"] = ne["Hospital Type"].map(TYPE).fillna("Other")
table = pd.crosstab(ne["State"], ne["type"])
table


In [ ]:
# --- Helpers: read them, do not change them ------------------------------------------
def lines(table):
    """Every row and every column of the table, each as a list of (row, column) cell addresses."""
    rows = [[(r, c) for c in table.columns] for r in table.index]
    cols = [[(r, c) for r in table.index] for c in table.columns]
    return rows + cols

def publish(table, mask):
    """The table as a reader would see it: hidden cells shown as *, totals always shown."""
    out = table.astype(str).where(~mask, "*")
    out["Total"] = table.sum(axis=1).astype(str)
    out.loc["Total"] = list(table.sum(axis=0).astype(str)) + [str(table.values.sum())]
    return out

def recover(table, mask):
    """Play the attacker. You see every unhidden cell and every total. A row or column with exactly
    one hidden cell gives it away: total minus the visible cells. Repeat until nothing new turns up.
    Returns {(row, column): value} for every cell recovered."""
    known = ~mask
    found = {}
    progress = True
    while progress:
        progress = False
        for line in lines(table):
            hidden = [cell for cell in line if not known.loc[cell]]
            if len(hidden) == 1:
                total = sum(table.loc[cell] for cell in line)
                visible = sum(table.loc[cell] for cell in line if known.loc[cell])
                found[hidden[0]] = int(total - visible)
                known.loc[hidden[0]] = True
                progress = True
    return found

def check(table, mask):
    """The release check. PASS means: no visible cell or total is 1-10, and the attacker recovers nothing."""
    problems = []
    shown = table.where(~mask)
    n_small = int(((shown >= 1) & (shown <= 10)).sum().sum())
    if n_small:
        problems.append(f"{n_small} visible cells are between 1 and 10")
    totals = list(table.sum(axis=1)) + list(table.sum(axis=0))
    if any(1 <= t <= 10 for t in totals):
        problems.append("a row or column total is between 1 and 10; collapse categories")
    leaked = recover(table, mask)
    if leaked:
        problems.append(f"the attacker recovers {len(leaked)} hidden cells: {leaked}")
    print("PASS" if not problems else "FAIL\n- " + "\n- ".join(problems))
    return not problems


In [ ]:
# TODO: primary = True where the cell is between 1 and 10 (inclusive), False elsewhere.
primary = ...

# print("cells hidden:", int(primary.values.sum()), "of", primary.size)
# publish(table, primary)


In [ ]:
# TODO: run the attacker on your primary suppression and print what it finds.


**What leaked, and how:** *(edit this cell)*

## 3. Complementary suppression
The fix is a second round: wherever a row or column has exactly one hidden cell, hide one more cell in the
same row or column, so the total no longer pins it down. By convention you hide the **smallest nonzero**
visible cell (you lose the least information; hiding a zero protects nothing, because a reader would
still know the hidden cell equals the total minus the visible cells). Hiding a new cell can leave
another row or column with a single hidden cell, so you repeat until nothing changes.

**Baseline:**
1. Finish `complementary` below (the loop is written; you write the three lines inside the `if`).
2. Run `check(table, final)`. It must print **PASS**.
3. Show `publish(table, final)` and report how many of the 24 cells are hidden in the end. One sentence:
   what would you do if this were your capstone table and too much disappeared?

In [ ]:
def complementary(table, mask):
    mask = mask.copy()
    changed = True
    while changed:
        changed = False
        for line in lines(table):
            hidden = [cell for cell in line if mask.loc[cell]]
            if len(hidden) == 1:
                # TODO 1: candidates = the cells in this line that are visible AND nonzero
                # TODO 2: pick the candidate with the smallest value (hint: min(candidates, key=lambda cell: table.loc[cell]))
                # TODO 3: hide it (mask.loc[pick] = True) and set changed = True
                pass
    return mask

# final = complementary(table, primary)
# check(table, final)
# print("cells hidden:", int(final.values.sum()), "of", final.size)
# publish(table, final)


**Published table, and what it cost:** *(edit this cell)*

## 4. De-identification under HIPAA Safe Harbor
HIPAA allows two ways to de-identify patient data (45 CFR 164.514(b)): **Expert Determination** (a
qualified statistician certifies the risk is very small) and **Safe Harbor**: remove 18 kinds of
identifiers, and have no actual knowledge that what is left could identify someone. The 18 are:
names; geography smaller than a state; all elements of dates except the year, and ages over 89;
telephone numbers; fax numbers; email addresses; Social Security numbers; medical record numbers; health
plan beneficiary numbers; account numbers; certificate/license numbers; vehicle identifiers; device
identifiers; URLs; IP addresses; biometric identifiers; full-face photographs; and any other unique
identifying number, characteristic, or code.

Three rules need care, quoted from the regulation (eCFR, 45 CFR 164.514(b)(2)(i)(B)–(C)):
- **ZIP codes.** You may keep the first three digits only if all ZIPs sharing them contain more than
  20,000 people; otherwise "the initial three digits ... is changed to 000." HHS's de-identification
  guidance (https://www.hhs.gov/hipaa/for-professionals/special-topics/de-identification/index.html)
  lists the 17 three-digit ZIPs that fail the test, using 2000 Census data. They are in `RESTRICTED_ZIP3`
  below. Note that 063, the ZIPs around New London, Connecticut, is one of them.
- **Dates.** "All elements of dates (except year)" go: birth, admission, discharge, death. Keep the year.
- **Ages.** "All ages over 89" are aggregated "into a single category of age 90 or older."

A record ID is allowed only if it is "not derived from or related to information about the individual"
(164.514(c)). A hash of the MRN is derived from the MRN. Use a fresh random number.

The cell below makes `raw`: 2,000 **synthetic** inpatient stays. **Baseline:** write `safe_harbor(raw)`
so it returns a file with exactly these columns: `record_id`, `zip3`, `age` (text: `"34"`, ..., `"89"`,
`"90+"`), `sex`, `admit_year`, `discharge_year`, `dx_code`, `diagnosis`, `readmit_30d`. Then
`check_safe_harbor(deid, raw)` must print **PASS**. Report how many records had their ZIP3 set to `000`
and how many ages became `90+`.

In [ ]:
# --- SYNTHETIC PATIENTS: NOT REAL PEOPLE -------------------------------------------
# Every name, date, ZIP code, phone number and record number here is made up by a random
# number generator with a fixed seed, so everyone in the class gets the same 2,000 rows.
# Phone numbers use the 555-01xx block and emails use example.com, both reserved for fiction.
import numpy as np

rng = np.random.default_rng(1212)
N = 2000
FIRST = ["Ana", "Ben", "Carla", "David", "Elena", "Frank", "Grace", "Hector", "Irene", "James", "Keisha",
         "Luis", "Maria", "Nathan", "Olivia", "Priya", "Quinn", "Rosa", "Samuel", "Tanya", "Victor", "Wen"]
LAST = ["Alvarez", "Brown", "Chen", "Diaz", "Evans", "Fischer", "Garcia", "Hughes", "Ivanova", "Johnson",
        "Kim", "Lopez", "Murphy", "Nguyen", "OBrien", "Patel", "Rossi", "Smith", "Torres", "Walsh"]
ZIPS = {"06824": .10, "06825": .08, "06604": .10, "06510": .10, "06106": .10, "06902": .10, "06810": .08,
        "06450": .08, "06226": .06, "06320": .05, "06360": .04, "06340": .03, "10282": .03, "03603": .03,
        "05901": .02}
DX = [("I50.9", "Heart failure", .18), ("J18.9", "Pneumonia", .16), ("J44.1", "COPD with exacerbation", .14),
      ("A41.9", "Sepsis", .12), ("N17.9", "Acute kidney failure", .10), ("I21.4", "NSTEMI", .10),
      ("E11.65", "Type 2 diabetes with hyperglycemia", .10), ("I63.9", "Cerebral infarction", .08),
      ("E84.0", "Cystic fibrosis with pulmonary manifestations", .01), ("D57.00", "Sickle cell crisis", .01)]

first, last = rng.choice(FIRST, N), rng.choice(LAST, N)
admit = pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, N), unit="D")
age_true = np.clip(rng.normal(66, 16, N).round(), 18, 102).astype(int)
dob = admit - pd.to_timedelta(age_true * 365.25 + rng.integers(1, 360, N), unit="D")
dx = rng.choice(len(DX), N, p=[d[2] for d in DX])
raw = pd.DataFrame({
    "mrn": rng.choice(np.arange(1_000_000, 10_000_000), N, replace=False).astype(str),
    "name": [f"{f} {l}" for f, l in zip(first, last)],
    "dob": dob.normalize(),
    "sex": rng.choice(["F", "M"], N),
    "phone": [f"203-555-01{n:02d}" for n in rng.integers(0, 100, N)],
    "email": [f"{f.lower()}.{l.lower()}{n}@example.com" for f, l, n in zip(first, last, rng.integers(1, 99, N))],
    "zip5": rng.choice(list(ZIPS), N, p=list(ZIPS.values())),
    "admit_date": admit,
    "discharge_date": admit + pd.to_timedelta(rng.integers(1, 12, N), unit="D"),
    "dx_code": [DX[i][0] for i in dx],
    "diagnosis": [DX[i][1] for i in dx],
    "readmit_30d": (rng.random(N) < 0.18).astype(int),
})
raw.head()


In [ ]:
# --- The release check for section 4: read it, do not change it --------------------
import hashlib

# HHS de-identification guidance: 3-digit ZIPs with 20,000 or fewer people (2000 Census).
RESTRICTED_ZIP3 = {"036", "059", "063", "102", "203", "556", "692", "790", "821",
                   "823", "830", "831", "878", "879", "884", "890", "893"}
KEEP = ["record_id", "zip3", "age", "sex", "admit_year", "discharge_year", "dx_code", "diagnosis", "readmit_30d"]

def check_safe_harbor(deid, raw):
    problems = []
    if list(deid.columns) != KEEP:
        problems.append(f"columns should be exactly {KEEP}, got {list(deid.columns)}")
    if len(deid) != len(raw):
        problems.append(f"{len(raw)} records in, {len(deid)} out")
    for c in deid.columns:
        if pd.api.types.is_datetime64_any_dtype(deid[c]) or deid[c].astype(str).str.contains(r"\d{4}-\d{2}-\d{2}").any():
            problems.append(f"{c} still holds full dates")
    if "zip3" in deid:
        if not deid["zip3"].astype(str).str.fullmatch(r"\d{3}").all():
            problems.append("zip3 must be exactly three digits, as text")
        if deid["zip3"].isin(RESTRICTED_ZIP3).any():
            problems.append("a restricted ZIP3 was kept; it must become 000")
    if "age" in deid:
        num = pd.to_numeric(deid["age"], errors="coerce")
        if (num > 89).any():
            problems.append("an age over 89 is still shown")
        if not set(deid["age"][num.isna()]) <= {"90+"}:
            problems.append("the only non-numeric age allowed is '90+'")
    if "record_id" in deid:
        ids = set(deid["record_id"].astype(str))
        hashes = {h(m.encode()).hexdigest() for m in raw["mrn"] for h in (hashlib.md5, hashlib.sha256)}
        if ids & set(raw["mrn"]) or ids & hashes:
            problems.append("record_id is the MRN or a hash of it; use a fresh random number")
        if deid["record_id"].duplicated().any():
            problems.append("record_id must be unique")
    print("PASS" if not problems else "FAIL\n- " + "\n- ".join(problems))
    return not problems


In [ ]:
def safe_harbor(raw):
    out = pd.DataFrame()
    # A fresh random ID, not derived from the MRN (seeded so your answer is reproducible):
    out["record_id"] = np.random.default_rng(7).permutation(len(raw)) + 1
    # TODO zip3: first three digits of zip5, but "000" for anything in RESTRICTED_ZIP3
    # TODO age: whole years from dob to admit_date (hint: (raw["admit_date"] - raw["dob"]).dt.days // 365.25),
    #           as text, with every age over 89 replaced by "90+"
    # TODO sex, admit_year, discharge_year (hint: .dt.year), dx_code, diagnosis, readmit_30d
    return out

deid = safe_harbor(raw)
check_safe_harbor(deid, raw)
# TODO: how many records got zip3 "000"? how many ages became "90+"?


**ZIP3s set to 000, ages 90+:** *(edit this cell)*

## 5. k-anonymity: what Safe Harbor does not promise
Safe Harbor is a checklist. It does not guarantee that a record is hidden in a crowd. **Quasi-identifiers**
are columns that are harmless alone but identifying together: ZIP3, age and sex are the classic three
(Latanya Sweeney showed in 2000 that ZIP, birth date and sex alone single out most Americans). A file is
**k-anonymous** if every combination of quasi-identifiers is shared by at least *k* records.

**Baseline:**
1. For `QI = ["zip3", "age", "sex"]`, count the records in each combination (`groupby(...).size()`).
   Report *k* (the smallest group) and how many records sit in groups smaller than 5.
2. Generalize age into the 10-year bands from `band()` and repeat with `["zip3", "age_band", "sex"]`.
3. Answer the two questions below.

In [ ]:
QI = ["zip3", "age", "sex"]
# TODO: sizes = number of records per QI combination; k = smallest; at_risk = records in groups < 5

def band(age):
    """'47' -> '40-49'; '90+' stays '90+'."""
    if age == "90+":
        return "90+"
    lo = int(age) // 10 * 10
    return f"{lo}-{lo + 9}"

# TODO: once safe_harbor() works: deid["age_band"] = deid["age"].map(band), then repeat with ["zip3", "age_band", "sex"]


**Safe Harbor passed, yet k is small. What does that mean, and who is most exposed?** *(edit this cell)*

**Your capstone data: does anything in it need suppression or de-identification before you publish? Which
columns or cells, and which rule applies?** *(edit this cell)*

## 6. Powerups
**(a) A model card for your week-6 model.** A model card (Mitchell et al., *Model Cards for Model
Reporting*, 2019) is a short document that travels with a model and says what it is for, what it is not
for, how well it works, and **for whom it works less well**. Fill in the template below for the model you
evaluated in week 6. Use your own numbers. Governance is the last section on purpose: a model nobody owns
and nobody re-checks is a liability, however good its AUROC.

### Model card: *(model name)*
*(edit this cell)*

**1. Model details.** Who built it, when, version, model type (e.g. logistic regression), the software used.

**2. Intended use.** The decision it supports and who makes that decision. The unit of prediction (here,
a hospital).

**3. Out-of-scope uses.** What it must not be used for, and why. Be specific.

**4. Data.** Source and release date of the training data, the target and how it was built, the
predictors, rows dropped and why, the train / test split.

**5. Performance.** AUROC, AUPRC and calibration on the hold-out, next to the baseline (base rate,
majority-class accuracy). Say whether the model beats the baseline by enough to matter.

**6. Subgroup performance.** The subgroup table from week 6. Name the subgroup where it does worst and
give the numbers with their uncertainty.

**7. Ethical considerations.** Who could be harmed by a wrong prediction, and how. Any proxy for a
protected characteristic among the predictors. Leakage checks done.

**8. Caveats and limitations.** What the public data cannot see. What would change the results.

**9. Governance.** Owner (a named person). Who may change it. When it is re-validated (e.g. every CMS
refresh). What triggers retirement. Where this card and the code live.